<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/Langchain_and_Unstructured_Data_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries and OpenAI API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change directory to this folder
%cd /content/drive/MyDrive/ai_agents_zero_to_mastery/RAG Bootcamp/RAG/Unstructured Data

In [ ]:
# Import the userdata module from Google Colab
from google.colab import userdata
# Retrieve the API key stored under 'genai_course' from Colab's userdata
api_key = userdata.get('ai_agent_openai')

In [ ]:
# Install libraries
!pip install langchain-community unstructured langchain-openai faiss-cpu -q
# Excel
!pip install msoffcrypto-tool -q
# Word
!pip install python-docx -q
!pip install --upgrade nltk -q
# PPT
!pip install python-pptx -q
# EPUB
!pip install pypandoc pandoc -q

# PDF
!pip install pymupdf pdfminer.six pillow_heif unstructured_inference unstructured_pytesseract pi_heif pdf2image -q
!apt-get install poppler-utils -q
!apt install tesseract-ocr -q

# Configurations

In [ ]:
# --- GLOBAL SETTINGS ---
# Embedding model
EMBEDDING_MODEL = "text-embedding-3-small"

# GPT model
GPT_MODEL = "gpt-5.4-nano"

In [ ]:
# set the api key in os
import os
os.environ["OPENAI_API_KEY"] = api_key

# Helpful functions

In [ ]:
# Import essential libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from IPython.display import Markdown, display

In [ ]:
# Prepare excel data
def prepare_excel(file_path):
  df = pd.read_excel(file_path)

  # One document / chunk per row
  docs_chunks = [
      Document(page_content = "\n".join(f"{col}: {val}" for col, val in row.items()),
              metadata = {"row_index": int(i)}
              )
      for i, row in df.iterrows()
  ]
  return docs_chunks

In [ ]:
# Prepare function for retrieval
def retrieval(
    docs_chunks_passed,
    embedding_model = EMBEDDING_MODEL,
    chunk_size_passed = 800,
    chunk_size_overlap_passed = 400):

  # Split into chunks
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size_passed,
      chunk_overlap = chunk_size_overlap_passed
  )
  chunks = text_splitter.split_documents(docs_chunks_passed)

  # Load the embedding model
  embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)

  # Create the vector store
  db_faiss = FAISS.from_documents(docs_chunks_passed, embeddings)

  return db_faiss

In [ ]:
def generation(db_faiss, query_passed, k = 20, model = GPT_MODEL):
  # Retrieve the context
  retrieved_docs = db_faiss.similarity_search_with_score(query_passed, k=k)
  context = "\n".join([doc.page_content for doc, score in retrieved_docs])

  # Build the prompt
  prompt = f"""
  answer the query {query_passed} based on the retrieved context: {context}.
  If you don't have the contet necessary, say that you do not know.
  """

  # Call the API
  model = ChatOpenAI(model = GPT_MODEL)
  answer = model.invoke(prompt)

  return answer

# Excel

In [ ]:
# Import libraries
import pandas as pd
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_core.documents import Document

In [ ]:
# Import the data
df = pd.read_excel("Reviews.xlsx")
df.head()

In [ ]:
# Test the functions
query_passed = "What are the most critical/negative reviews?"
chunks_excel = prepare_excel("Reviews.xlsx")
db_faiss = retrieval(chunks_excel, embedding_model=EMBEDDING_MODEL)
answer = generation(db_faiss, query_passed, k = 20, model = GPT_MODEL)

In [ ]:
display(Markdown(answer.content))

### Word

In [ ]:
# Import necessary libraries for word documents
import nltk
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
nltk.download('punkt')

In [ ]:
def prepare_word(file_path):
  # Load the word document into docs
  loader = UnstructuredWordDocumentLoader(file_path)
  docs_word = loader.load()
  return docs_word

In [ ]:
query_passed = "What is the main point of this document?"

# Load the word document into docs
docs_word = prepare_word("slide style design.docx")

# Apply the retrieval (text splitting and get database vectorstore for similarity search)
db_faiss = retrieval(docs_word, embedding_model=EMBEDDING_MODEL)

# Apply generation
answer = generation(db_faiss, query_passed, k = 20, model = GPT_MODEL)

In [ ]:
display(Markdown(answer.content))

### ppt

In [ ]:
# Essential class
from langchain_community.document_loaders import UnstructuredPowerPointLoader

In [ ]:
# Build a function to prepare the data
def prepare_ppt(file_path):
  loader = UnstructuredPowerPointLoader(file_path)
  docs_ppt = loader.load()
  return docs_ppt

docs_ppt = prepare_ppt("6.12 Working with PowerPoint Presentations Using Langchain.pptx")

In [ ]:
# Prepare the VDB
db_faiss_ppt = retrieval(docs_ppt, chunk_size_passed=200, chunk_size_overlap_passed=20)

In [ ]:
# Generate the answer
answer = generation(db_faiss_ppt, "What is this lecture about?")

In [ ]:
display(Markdown(answer.content))

## EPUB

In [ ]:
# Import essential module
from langchain_community.document_loaders import UnstructuredEPubLoader
import pypandoc
pypandoc.download_pandoc()

In [ ]:
# Build the function
def prepare_epub(file_path):
  loader = UnstructuredEPubLoader(file_path)
  docs_epub = loader.load()
  return docs_epub

docs_epub = prepare_epub("fitzgerald-great-gatsby.epub")

In [ ]:
# Prepare the db
db_faiss_epub = retrieval(docs_epub)

In [ ]:
# Answer the query
answer = generation(db_faiss_epub, "What is the main point of this book?")

In [ ]:
display(Markdown(answer.content))

## PDF

In [ ]:
# Import modules
from langchain_community.document_loaders import UnstructuredPDFLoader
from pdfminer import psparser

In [ ]:
# Build function for the PDF
def prepare_pdf(file_path):
  loader = UnstructuredPDFLoader(file_path, language = "EN")
  docs = loader.load()
  return docs

In [ ]:
docs_pdf = prepare_pdf("airbnb-original-deck-2008.pdf")

In [ ]:
# Build de DB
db_faiss_pdf = retrieval(docs_pdf)

In [ ]:
# Generate the answer
answer = generation(db_faiss_pdf, "Why is Airbnb a good investmen?")

In [ ]:
display(Markdown(answer.content))